# Results Presentation
Polished notebook for internship presentation.

---
**Author:** Internship Project | **Date:** 2026  
**Model:** YOLOv8m | **Task:** Ice cream vending machine shelf analysis


## 1. Project Overview
This project automates the manual process of vending machine inventory checking. A camera captures a shelf photo, and our YOLOv8-based pipeline:
1. Detects every ice cream product
2. Classifies it by brand/flavour
3. Counts quantities per category
4. Computes shelf fill rate
5. Flags restocking needs
6. Displays results in a real-time dashboard


In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import yaml

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)
CLASS_NAMES = CFG['classes']

MODEL_PATH = None  # replace with your best.pt
print('Project:', CFG['project']['name'])
print('Version:', CFG['project']['version'])
print('Classes:', len(CLASS_NAMES))

## 2. Dataset Statistics

In [ ]:
LABELS_DIR = Path('../data/annotations')
class_counts = Counter()
for lbl in LABELS_DIR.glob('*.txt'):
    for line in lbl.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            class_counts[int(parts[0])] += 1

if class_counts:
    labels = [CLASS_NAMES[k] for k in sorted(class_counts)]
    counts = [class_counts[k] for k in sorted(class_counts)]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(labels, counts, color='#4CAF50')
    ax.set_title('Training Data: Annotation Count per Class', fontsize=14)
    ax.set_xlabel('Ice Cream Category')
    ax.set_ylabel('Count')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No label files found yet — run annotation first.')

## 3. Training Summary

In [ ]:
RUNS_DIR = Path('../runs')
all_csvs = sorted(RUNS_DIR.rglob('results.csv'), reverse=True)

if all_csvs:
    df = pd.read_csv(all_csvs[0])
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if 'metrics/mAP50(B)' in df.columns:
        axes[0].plot(df['metrics/mAP50(B)'], color='#2196F3', linewidth=2)
        axes[0].set_title('mAP@0.5 over Training', fontsize=13)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylim(0, 1)
    if 'train/box_loss' in df.columns:
        axes[1].plot(df['train/box_loss'], color='#F44336', linewidth=2, label='Train')
        if 'val/box_loss' in df.columns:
            axes[1].plot(df['val/box_loss'], color='#FF9800', linewidth=2, linestyle='--', label='Val')
        axes[1].set_title('Box Loss over Training', fontsize=13)
        axes[1].set_xlabel('Epoch')
        axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('No training runs found yet.')

## 4. Key Results

In [ ]:
REPORT_PATH = Path('../reports/evaluation_report.json')
if REPORT_PATH.exists():
    with open(REPORT_PATH) as f:
        report = json.load(f)
    overall = report['overall']
    print('=== Key Results ===')
    print(f"mAP@0.5        : {overall['mAP50']:.4f}")
    print(f"mAP@0.5:0.95   : {overall['mAP50_95']:.4f}")
    print(f"Counting Acc   : {overall['counting_accuracy']:.2%}")
    print(f"Mean Count Err : {overall['mean_absolute_count_error']:.2f}")
else:
    print('Evaluation report not found. Run scripts/evaluate.sh first.')

## 5. Live Demo

In [ ]:
from src.models.predictor import predict_shelf
from src.utils.visualization import draw_detections

DEMO_IMAGE = None  # set to an image path, e.g. '../data/splits/images/test/shelf_001.jpg'

if DEMO_IMAGE:
    result = predict_shelf(DEMO_IMAGE, MODEL_PATH, conf_threshold=0.5)
    annotated = result['annotated_image']
    original  = cv2.imread(DEMO_IMAGE)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.imshow(cv2.cvtColor(original,  cv2.COLOR_BGR2RGB))
    ax1.set_title('Original', fontsize=13)
    ax1.axis('off')

    ax2.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax2.set_title(f"Detected | Fill rate: {result['shelf_fill_rate']:.0%}", fontsize=13)
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

    print('Summary:')
    for cat, info in result['summary'].items():
        print(f"  {cat:30s}  count={info['count']}  conf={info['avg_confidence']:.2f}")
else:
    print('Set DEMO_IMAGE to a shelf image path to run the demo.')

## 6. Challenges & Solutions

| Challenge | Solution |
|-----------|----------|
| Limited real shelf data | Prototype web scraping + aggressive augmentation |
| Glass reflections on vending machine door | Random shadow augmentation |
| Similar-looking product variants (e.g. Magnum Classic vs White) | Per-class hard examples; larger model (YOLOv8m) |
| Empty slot detection (unusual for YOLO) | Explicit `empty_slot` class with bounding boxes |
| Inference speed on edge devices | YOLOv8n variant for real-time deployment |

## 7. Future Work

- **Price tag OCR** — read price labels and link to inventory system
- **Temporal tracking** — compare shelves over time to detect sell-through rates
- **Planogram compliance** — verify products are in the correct shelf positions
- **Mobile app** — lightweight YOLOv8n on-device inference via ONNX / CoreML
- **Auto-restock alerts** — integration with supply chain / warehouse system
